In [ ]:
from vascx.retina import Retina
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from PIL import Image
from scipy.ndimage import distance_transform_edt
from vascx.analysis.network import get_segments

In [ ]:
r = Retina.from_file('../samples/av_hrf/01_h_new.png')
# r.load_fundus_image('./samples/av_hrf/images/01_h.jpg')

In [ ]:
segments = r.veins.segments

In [ ]:
np.mean(segments[0].pixels, axis=0)

In [ ]:
len(segments)

In [ ]:
segments[51].get_diameters()

In [ ]:
r.veins.plot_segments()

In [ ]:
r.plot()

In [ ]:
img = r.veins.skeleton.astype(np.uint8)*255

In [ ]:
x_closest, y_closest = distance_transform_edt(~img, return_distances=False, return_indices=True)

In [ ]:
x_closest.shape

In [ ]:
len(r.veins.segments)

In [ ]:
r.plot(layers=['arteries'])

In [ ]:
r = Retina.from_file('../samples/av_hrf/02_g.png')
r.load_disc('../samples/av_hrf/disc/02_g.png')

In [ ]:
skeleton_pixel_to_segment = {
    pixel: seg for seg in r.veins.segments for pixel in seg.pixels
}

In [ ]:
dt = distance_transform_edt(~r.veins.skeleton, return_distances=False, return_indices=True)

In [ ]:
plt.imshow(distance_transform_edt(~r.veins.skeleton))

In [ ]:
dt.shape

In [ ]:
nz = np.nonzero(r.veins.binary)

In [ ]:
np.min(nz[0]), np.max(nz[0])

In [ ]:
np.min(nz[1]), np.max(nz[1])

In [ ]:
for y, x in zip(*np.nonzero(r.veins.binary)):
    closest_point = tuple(dt[:,y,x].squeeze())
    # print(closest_point)
    if closest_point in skeleton_pixel_to_segment:
        segment = skeleton_pixel_to_segment[closest_point]
        segment.pixels.append((y,x))

In [ ]:
r.plot()

In [ ]:
len(r.veins.segments)

In [ ]:
r.veins.plot_segments()

In [ ]:
source_segments = [seg for seg in r.veins.segments if any([r.disc_mask[e] for e in seg.ends])]
for segment in r.veins.segments:
    segment.neighbors = [seg for seg in r.veins.segments if segment != seg and len(segment.connectors.intersection(seg.connectors)) > 0]

In [ ]:
def recursive_set_rank(seg, value=1):
    seg.rank = 1
    for s in seg.neighbors:
        if s.rank != 1:
            recursive_set_rank(s, value)

In [ ]:
for seg in source_segments:
    recursive_set_rank(seg)

In [ ]:
r.veins.plot_segments(rank=1)

In [ ]:
Image.fromarray(r.veins.skeleton).save('output.png')

In [ ]:
from vascx.analysis.network import make_adjacency_graph_8, connect_segment
from vascx.segment import Segment

In [ ]:
def plot_skl(skeleton, segments):
    arr = np.repeat((skeleton*255).astype(np.uint8)[:, :, None], 3, axis=2)

    for segment in segments:
        for pix in segment.pixels:
            arr[pix[0],pix[1],:] = [255,0,0]
        for pix in segment.ends:
            arr[pix[0],pix[1],:] = [0,255,0]
        for pix in segment.connectors:
            arr[pix[0],pix[1],:] = [0,0,255]


    # print(np.unique(arr))
    im = Image.fromarray(arr)
    # im = skeleton
    
    
    # # im = np.ma.masked_where(im == 0, im)


    # # ax.imshow(np.zeros_like(im))
    # ax.imshow(im, cmap='viridis')

    # return fig, ax
    return im

In [ ]:

def explore(graph, vertex, visited):

    pixels = set()
    ends = set()
    connectors = set()
    queue = [vertex]
    while queue:
        current = queue.pop()
        visited.add(current)

        pixels.add(current)
        neighbors = graph[current]

        if len(neighbors) > 2:
            ends.add(current)

            connectors.add(current)
            connectors.update(neighbors)
            continue

        has_neighbor = False
        for n in neighbors:
            if n not in visited:
                has_neighbor = True
                queue.append(n)

        if not has_neighbor:
            ends.add(current)

            connectors.add(current)
            connectors.update(neighbors)
    return pixels, ends, connectors

In [ ]:
def explore_n(n):
    pixels = set(zip(*np.where(r.veins.skeleton)))
    graph = make_adjacency_graph_8(pixels)

    segments = []
    visited = set()

    for vertex in list(graph.keys())[:n]:
        if vertex not in visited:
            
            pixels, ends, connectors = explore(graph, vertex, visited)
            segments.append(Segment(connect_segment(pixels, ends), frozenset(
                ends), frozenset(connectors - pixels)))

    return plot_skl(r.veins.skeleton, segments)

In [ ]:
explore_n(2).save('output.png')